In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:

EMBEDDING_MODEL='text-embedding-3-large'
GPT_MODEL='gpt-5.4'

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.vectorstores import FAISS
api_key=os.getenv('OPENAI_API_KEY')

#import essential libraries
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_community.vectorstores import FAISS
from IPython.display import display, Markdown

/var/folders/59/dd24b3q96ss394n23gj0nv840000gn/T/ipykernel_13439/1879386965.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [1]:
from langchain_community.document_loaders import UnstructuredPDFLoader
from pdfminer import psparser

In [6]:
#define the path to the PDF file
def prepare_pdf(file_path):
    loader = UnstructuredPDFLoader(file_path,language='eng')
    docs = loader.load()
    return docs

docs_pdf= prepare_pdf('Data/airbnb-original-deck-2008.pdf')


No languages specified, defaulting to English.


In [7]:
#Prepare the function for retrieval
def retrieval(docs,embedding_model='text-embedding-3-large'):
    #load the embedding model
    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
    #store in the vector database
    df_faiss = FAISS.from_documents(chunks, embeddings)
    return df_faiss

#Prepare the function for generation
def generation(df_faiss, query,k=20,gpt_model='gpt-5.4'):
    #Retrieve the content
    retrieved_docs=df_faiss.similarity_search_with_score(query,k=20)
    context="\n".join([doc.page_content for doc,score in retrieved_docs])
    
    #build the prompt for the LLM
    prompt=f"""
    answer the query {query} based on the retrieved context {context} below.
    If the answer is not contained within the context, say "I don't know".
    """
    #call the API
    model=ChatOpenAI(model_name=GPT_MODEL,temperature=0.2,openai_api_key=api_key)
    answer=model.invoke(prompt)
    return display(Markdown(answer.content))

In [ ]:
#Build the DB
db_faiss=retrieval(docs_pdf)

